# 🚀 StoryDiffusion x Agentic Comic Generator
Notebook này import trực tiếp **kiến trúc LangGraph** từ dự án của bạn (Director, Writer, Storyboarder, Validator) và chỉ thay thế node `Renderer` bằng thuật toán **StoryDiffusion** để giữ khuôn mặt đồng nhất 100%!

> **Lưu ý:** Vui lòng bật GPU (T4 hoặc A100) trước khi chạy.

In [ ]:
!pip install -qU diffusers transformers accelerate langchain langchain-openai langgraph pydantic
import sys
import os
sys.path.append('.') # Để import các module từ repo hiện tại
print('✅ Environment Setup Complete!')

In [ ]:
import os
import json
import torch
import copy
from PIL import Image
import matplotlib.pyplot as plt
from langgraph.graph import StateGraph, END

# ==========================================
# 🔑 ĐIỀN OPENAI API KEY CỦA BẠN VÀO ĐÂY
os.environ['OPENAI_API_KEY'] = 'sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'
# ==========================================

# IMPORT AGENTS TỪ REPO HIỆN TẠI
from studio_graph.agents import StudioState, run_director, run_writer, run_storyboarder, run_validator

print('✅ Đã load thành công Agents từ repo hiện tại!')

In [ ]:
from diffusers import StableDiffusionXLPipeline, DDIMScheduler
from sd_utils.gradio_utils import AttnProcessor2_0 as AttnProcessor
from sd_utils.gradio_utils import cal_attn_mask_xl
from sd_utils.gradio_utils import SpatialAttnProcessor2_0
import sd_utils.gradio_utils as gradio_utils

device = 'cuda' if torch.cuda.is_available() else 'cpu'
sd_model_path = 'SG161222/RealVisXL_V4.0'

print('⏳ Loading SDXL Pipeline...')
pipe = StableDiffusionXLPipeline.from_pretrained(sd_model_path, torch_dtype=torch.float16, use_safetensors=True)
pipe = pipe.to(device)
pipe.enable_freeu(s1=0.6, s2=0.4, b1=1.1, b2=1.2)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.scheduler.set_timesteps(50)

def setup_storydiffusion(pipe, num_panels, id_length=3):
    unet = pipe.unet
    attn_procs = {}
    for name in unet.attn_processors.keys():
        cross_attention_dim = None if name.endswith('attn1.processor') else unet.config.cross_attention_dim
        if name.startswith('mid_block'):
            hidden_size = unet.config.block_out_channels[-1]
        elif name.startswith('up_blocks'):
            block_id = int(name[len('up_blocks.')])
            hidden_size = list(reversed(unet.config.block_out_channels))[block_id]
        elif name.startswith('down_blocks'):
            block_id = int(name[len('down_blocks.')])
            hidden_size = unet.config.block_out_channels[block_id]
        if cross_attention_dim is None and name.startswith('up_blocks'):
            attn_procs[name] = SpatialAttnProcessor2_0(id_length=id_length)
        else:
            attn_procs[name] = AttnProcessor()
    unet.set_attn_processor(copy.deepcopy(attn_procs))
    gradio_utils.mask1024, gradio_utils.mask4096 = cal_attn_mask_xl(num_panels, id_length, 0.5, 0.5, 768, 768, device=device, dtype=torch.float16)
    gradio_utils.total_count = sum(1 for name in unet.attn_processors.keys() if name.startswith('up_blocks') and not name.endswith('attn1.processor'))
    gradio_utils.attn_count = 0
    gradio_utils.id_length = id_length
    gradio_utils.total_length = num_panels
    gradio_utils.cur_step = 0
    print('✅ Khởi tạo StoryDiffusion xong!')


In [ ]:
def run_storydiffusion_renderer(state: StudioState) -> StudioState:
    print('\n[STORY-DIFFUSION RENDERER] Bắt đầu vẽ các khung truyện...')
    schema = state['current_schema']
    panels = schema.get('panels', [])
    
    # Tạo mảng prompt dựa vào schema (Character + Panel description)
    # Ở đây chúng ta extract bối cảnh từ script của Storyboarder
    prompts = [p.get('description', '') for p in panels]
    if not prompts:
        print('Không có panels nào để vẽ!')
        return state
        
    setup_storydiffusion(pipe, num_panels=len(prompts))
    gradio_utils.attn_count = 0
    gradio_utils.cur_step = 0
    generator = torch.Generator(device=device).manual_seed(42)
    
    print(f'🎨 Đang vẽ {len(prompts)} khung tranh đồng thời...')
    out = pipe(prompt=prompts, height=768, width=768, num_inference_steps=25, generator=generator, guidance_scale=5.0)
    
    # Hiển thị ảnh
    fig, axes = plt.subplots(1, len(prompts), figsize=(15, 5))
    if len(prompts) == 1:
        axes = [axes]
    for ax, img, p in zip(axes, out.images, prompts):
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(p[:30] + '...', fontsize=9)
    plt.tight_layout()
    plt.show()
    
    state['current_page_idx'] += 1
    return state

# BUILD GRAPH
workflow = StateGraph(StudioState)
workflow.add_node('director', run_director)
workflow.add_node('writer', run_writer)
workflow.add_node('storyboarder', run_storyboarder)
workflow.add_node('validator', run_validator)
workflow.add_node('renderer', run_storydiffusion_renderer) # <--- NODE MỚI!

workflow.set_entry_point('director')
workflow.add_edge('director', 'writer')
workflow.add_edge('writer', 'storyboarder')
workflow.add_edge('storyboarder', 'validator')

def validator_router(state: StudioState):
    return state['next_step']
workflow.add_conditional_edges('validator', validator_router, {'renderer': 'renderer', 'storyboarder': 'storyboarder'})

def renderer_router(state: StudioState):
    if state['current_page_idx'] >= len(state.get('page_scripts', [])):
        return 'end'
    return 'storyboarder'
workflow.add_conditional_edges('renderer', renderer_router, {'end': END, 'storyboarder': 'storyboarder'})

app = workflow.compile()
print('✅ Đã nối Graph xong với StoryDiffusion!')

In [ ]:
# CHẠY GRAPH!
idea = 'Một nữ thợ săn tiền thưởng không gian tóc đỏ, cầm súng laser, săn lùng quái vật trên hành tinh hoang dã.'
inputs = {'user_prompt': idea}

print('🚀 Bắt đầu tạo truyện...')
config = {'configurable': {'thread_id': 'colab_test_1'}}
for output in app.stream(inputs, config=config, stream_mode='values'):
    pass # Log đã được print sẵn trong các node